## Notebook for plotting 2d histograms of STICS PHA data read from tab, comma or space delimited text files

In [ ]:
import csv
import datetime as dt
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LogNorm
from matplotlib.figure import Figure

from geotail_cmap import geotail_cmap

%matplotlib inline

In [ ]:
# functions for reading STICS data from excel or text files

def read_excel_workbook(filename) -> pd.DataFrame:
    # read the binary excel workbook at filename and return a dataframe with a
    # DateTimeIndex from the #MPHA.GSTICS.IEEETimeStr column and float columns
    # of MPQ and MASS
    return (
        pd.read_excel(filename, header=0, usecols=[0, 1, 2], )
        .dropna()
        .rename(columns={'#MPHA.GSTICS.IEEETimeStr': 'MPHA_GSTICS_IEEETimeStr'})
        .query('MPHA_GSTICS_IEEETimeStr.str[0] != "#"')
        .set_index('MPHA_GSTICS_IEEETimeStr')
        .pipe(lambda df_: df_.set_index(pd.to_datetime(df_.index, format='%Y-%jT%H:%M:%S.%f')))
    )


def read_text_data(filepath: Path|str) -> pd.DataFrame:
    # read the text file with pandas.read_csv()  The file can be comma delimited,
    # tab delimited, or contain columns separated by spaces
    # return a dataframe with the STICS PHA data
    # Use the csv package to determine the delimiter
    with open(filepath, 'r', newline='') as stics_file:
        dialect = csv.Sniffer().sniff(stics_file.read(1024), delimiters='\t, ')
    sep = dialect.delimiter
    if sep == ' ':
        sep = r' +'
    return pd.read_csv(filepath, sep=sep)


In [ ]:

def set_tick_params(ax):
    ax.tick_params(axis="x", which='major', length=7, labelsize=11, reset=True)
    ax.tick_params(axis="y", which='major', length=7, labelsize=11, reset=True)
    ax.tick_params(axis="x", which='minor', length=5, labelsize=10, reset=True)
    ax.tick_params(axis="y", which='minor', length=4, labelsize=10, reset=True)


def make_2d_hist(df: pd.DataFrame,
                 xval: str='MPQ',
                 yval: str='MASS',
                 logxy: bool=True,
                 xlo: float=0.5,
                 xhi: float=100,
                 ylo: float=0.5,
                 yhi: float=100,
                 bin_width: float=.01,
                 ) -> tuple[np.array]:
    if isinstance(bin_width, (tuple, list)):
        xbinw = bin_width[0]
        ybinw = bin_width[1]
    else:
        xbinw = ybinw = bin_width
    if logxy:
        num = int((np.log10(xhi) - np.log10(xlo)) / xbinw) + 1
        xbins = np.geomspace(xlo, xhi, num=num)
        ybins = np.geomspace(ylo, yhi, num=num)
    else:
        xbins = np.arange(xlo, xhi + xbinw, xbinw)
        ybins = np.arange(ylo, yhi + ybinw, ybinw)

    hist2d, xedges, yedges = np.histogram2d(
        x=df[xval],
        y=df[yval],
        bins=[xbins, ybins],
        range=[[xlo, xhi + xbinw], [ylo, yhi + ybinw]],
        # weights=df['Weights'],
    )

    return xbins, ybins, hist2d.T


def plot_2d_hist(
        df: pd.DataFrame,
        xlo: float=0.5,
        xhi: float=1e2,
        ylo: float=0.5,
        yhi: float=1e2,
        zmin: float=1.,
        zmax: float=0.,
        tic: int = 0,
        bin_width: float = 0.01,
        xval: str = 'MPQ',
        xlabel: str = '',
        yval: str = 'MASS',
        ylabel: str = '',
        title: str = '',
        cmap=None,
        withcb: bool = True,
        grid: bool = False,
        logxy: bool = True,
        fig: Figure = None,
) -> Figure:
    """
    Create 2D plots of xval, yval PHA data
    :param xval: dataFrame column to use for x values.  Default is TOF
    :param xlabel: string to use for the x axis label.  Default is '' which uses xavl for the label
    :param yval: dataFrame column to use for y values.  Default is Energy
    :param ylabel: string to use for the y axis label.  Default is '' which uses yavl for the label
    :param title: The plot title. Pass in the empty string to generate the title automatically.
    :param cmap: matplotlib colormap, defaults to the cmap in Steve's GEOTAIL plots
    :param withcb: add a colorbar to the plot.  Default is True
    :param grid: draw a grid if True.  Default is False
    :param logxy: use log x and y axes if True.  Default is False
    :param fig: matplotlib figure for plotting.  Default is None to force this routine to create the figure
    :return: matplotlib.figure.Figure
    """
    mpl.rcParams['axes.titlepad'] = 12
    mpl.rcParams['axes.titlesize'] = 13
    mpl.rcParams['axes.labelsize'] = 14
    mpl.rcParams['axes.formatter.min_exponent'] = 4
    xbins, ybins, hist2d = make_2d_hist(df=df, xval=xval, yval=yval, logxy=logxy)
    cmap = cmap or geotail_cmap
    if fig is None:
        plt.close('all')
        fig = Figure(figsize=[13.75, 11],
                     tight_layout=True,
                     dpi=72)
        ax = fig.add_subplot(111,
                             xlim=[xlo, xhi],
                             ylim=[ylo, yhi],
                             xlabel=xlabel or xval,
                             ylabel=ylabel or yval,
                             title=title or title,
                             )
        set_tick_params(ax)
    else:
        ax = fig.axes[0]
    if logxy:
        ax.set_xscale("log")  # <- Activate log scale on X axis
        ax.xaxis.set_minor_formatter(mpl.ticker.LogFormatter(base=10, labelOnlyBase=False, minor_thresholds=(3, 2)))
        ax.set_yscale("log")  # <- Activate log scale on Y axis
        ax.yaxis.set_minor_formatter(mpl.ticker.LogFormatter(base=10, labelOnlyBase=False, minor_thresholds=(3, 2)))
    else:
        if tic > 0:
            ax.set_xticks(np.arange(xlo, xhi + tic, tic))
            ax.set_yticks(np.arange(ylo, yhi + tic, tic))
    match (zmin, zmax):
        case (1, 0):
            norm = LogNorm()
        case (_, 0):
            norm = LogNorm(vmin=zmin)
        case (1, _):
            norm = LogNorm(vmax=zmax)
        case _:
            norm = LogNorm(vmin=zmin, vmax=zmax)
    pcm = ax.pcolormesh(
        xbins, ybins, hist2d, #np.where(hist2d >= zmin, hist2d, np.nan),
        norm=norm, shading='auto', cmap=cmap, rasterized=True
    )
    if withcb:
        cb = fig.colorbar(pcm, ax=ax, aspect=30)  # , shrink=0.8)
        cb.set_label('# PHAs', fontsize=16)
        cb.ax.tick_params(axis="y", which='major', length=10, labelsize=14)
        cb.ax.tick_params(axis="y", which='minor', length=5)
    if grid:
        ax.grid(True, color='lightgray', which='both' if logxy else 'major')
        ax.set_axisbelow(True)
    # ax.figure.text(0.72, 0.065, f'Plotted {dt.date.today()}', fontsize=10)
    ax.figure.text(0.79, 0.02, f'Plotted {dt.date.today()}', fontsize=10)
    return fig


In [ ]:
stics_file = Path('./ESM_1993012033TOR<6M>6edit.tsv')
df = read_text_data(stics_file)
fig = plot_2d_hist(df, xlo=4, xhi=100, ylo=4, yhi=100, 
                   title=stics_file.with_suffix('.qda'))
fig
fig.savefig(stics_file.with_suffix('.qda.png'))


In [ ]:
stics_file = Path('ESM_1993012,033TOR<6*_o.csv')
df = read_text_data(stics_file)
fig = plot_2d_hist(df, xlo=.5, xhi=100, ylo=.5, yhi=100, yval='MASS2', 
                   title=stics_file.with_suffix('.qda'))
fig
fig.savefig('ESM_1993012,033TOR<6*_o.qda.png')


In [ ]:
stics_file = Path('ESM_1993012,033TOR<6M>6.xlsb')
exceldf = read_excel_workbook(filename=stics_file)
fig = plot_2d_hist(exceldf, xlo=.5, xhi=100, ylo=.5, yhi=100, 
                   title=stics_file.name)
fig
fig.savefig(stics_file.with_suffix('.xlsb.png'))
